# MNN dự báo quỹ đạo bão: quét hệ số chính quy hoá L2

Notebook này dùng đúng mô hình, dữ liệu và cách chia tập của dự án (phần 0–5, giống `MNN_TongHop.ipynb`), rồi ở **phần 6** huấn luyện lại với nhiều giá trị **L2 λ** để xem phạt L2 lên trọng số W_L, W_R có làm giảm sai số không.

- Hàm mất mát khi huấn luyện: MSE + λ·Σ‖W‖²/2. Loss báo cáo, chọn checkpoint và dừng sớm vẫn là MSE thuần.
- Mỗi λ chạy nhiều seed; báo cáo Mean, **RMSE**, Median, P95 sai số Haversine (km), so với λ = 0 kèm CI95 bootstrap theo cơn bão.
- **Chọn λ theo tập val**, tập test chỉ để báo cáo.

**Cách chạy:** đặt `ibtracs_tbd_6h.csv` cùng thư mục (hoặc thư mục cha) rồi *Run All*. Kết quả từng lần chạy được lưu trong `KetQua_L2/`; chạy lại sẽ bỏ qua những lần đã có.

## Thư viện

Notebook dùng `numpy`, `pandas`, `matplotlib`, `scikit-learn` và `tensorflow` (bản 2.x, gọi qua `tf.compat.v1`).
Chạy ô dưới để xem phiên bản đang dùng:

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # tránh crash kernel do trùng thư viện OpenMP trên Windows

import time
import random
import shutil

import numpy as np
import tensorflow as tf2              # import TF trước matplotlib/sklearn
import tensorflow.compat.v1 as tf
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

tf.disable_v2_behavior()
tf2.get_logger().setLevel('ERROR')

plt.rc('font', **{'size': 12})
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)

for name, mod in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib),
                  ('sklearn', sklearn), ('tensorflow', tf2)]:
    print(f'{name:11s} {mod.__version__}')

## 0. Cấu hình chung

Các tham số giống `MNN_Main.py`. `QUICK = True` chạy nhanh 4 epoch, 1 seed để xem xu hướng; số liệu đáng tin cần `QUICK = False` (tối đa 200 epoch, dừng sớm).

In [ ]:
SEED = 42
CSV_PATH = 'ibtracs_tbd_6h.csv'      # tự tìm ở thư mục notebook, rồi thư mục cha
for _p in [CSV_PATH, os.path.join('..', CSV_PATH)]:
    if os.path.exists(_p):
        CSV_PATH = _p
        break
print('Dùng dữ liệu:', os.path.abspath(CSV_PATH))

FORECAST_LIST = [1, 2]               # số bước 6h: 1 -> +6h, 2 -> +12h
WINDOW_SIZE = 4                      # 4 bước = 24h quá khứ
FIRST_HID = [2, 25, 8]               # lớp ẩn 1: ma trận 25 x 8
SECOND_HID = [2, 50, 10]             # lớp ẩn 2: ma trận 50 x 10
TWO_STAGE = False
MAX_EPOCHS = 200
STOP_PATIENCE = 30

MAX_STEP_KM = 600.0                  # cắt đoạn nếu nhảy > 600 km / 6h
MIN_SEG_LEN = WINDOW_SIZE + max(FORECAST_LIST) + 1
NATURE_KEEP = None
MIN_MOVE_KM = 10.0

KM_PER_DEG = 111.195
COL_LAT_ABS = 5                      # cột 5 của bảng = vĩ độ tuyệt đối

QUICK = False
if QUICK:
    MAX_EPOCHS = 4

random.seed(SEED)
np.random.seed(SEED)

## 1. Dữ liệu

Tệp `ibtracs_tbd_6h.csv` có một dòng cho mỗi mốc 6h của mỗi cơn bão. Các cột dùng tới: `ID`, `ISO_TIME`, `LAT`, `LON`, `STORM_SPEED` (knot), `STORM_DIR` (độ), `NAME`, `YEAR`, `BASIN`, `NATURE`.

In [ ]:
raw = pd.read_csv(CSV_PATH, encoding='utf-8', low_memory=False, keep_default_na=False, na_values=[''])
print(raw.shape)
raw.head()

### 1.1 Cắt phân đoạn và tạo bảng cho từng phân đoạn

Mỗi cơn bão được cắt thành các phân đoạn liên tục: cắt ở chỗ hụt mốc 6h hoặc chỗ nhảy hơn `MAX_STEP_KM`. Phân đoạn ngắn hơn `MIN_SEG_LEN` điểm bị bỏ.

Mỗi phân đoạn thành một bảng 6 cột:

| Cột | Nội dung |
|---|---|
| 0 | kinh độ tương đối so với điểm đầu phân đoạn (độ) |
| 1 | vĩ độ tương đối (độ) |
| 2 | `STORM_SPEED` (knot) |
| 3, 4 | sin, cos của `STORM_DIR` |
| 5 | vĩ độ tuyệt đối |

In [ ]:
def haversine_distance(coords_true, coords_pred):
    """Khoảng cách Haversine (km); mỗi hàng là (lon, lat) theo độ."""
    R = 6371.0
    lon1, lat1 = np.radians(coords_true[:, 0]), np.radians(coords_true[:, 1])
    lon2, lat2 = np.radians(coords_pred[:, 0]), np.radians(coords_pred[:, 1])
    a = (np.sin((lat2 - lat1) / 2.0) ** 2 +
         np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def load_csv_data(csv_path, step_hours=6):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy file '{csv_path}'!")

    df = pd.read_csv(csv_path, encoding='utf-8', low_memory=False, keep_default_na=False, na_values=[''])
    df['BASIN'] = df['BASIN'].fillna('NA')
    df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'], errors='coerce')
    for c in ['STORM_SPEED', 'STORM_DIR', 'LAT', 'LON']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df[['STORM_SPEED', 'STORM_DIR']] = df.groupby('ID', sort=False)[['STORM_SPEED', 'STORM_DIR']].ffill().fillna(0.0)
    df = df.dropna(subset=['ISO_TIME', 'LAT', 'LON'])
    if NATURE_KEEP is not None:
        df = df[df['NATURE'].isin(NATURE_KEEP)]

    cyclones_data, initial_coords, meta = [], [], []
    n_jump_cuts = 0

    for storm_id, group in df.groupby('ID', sort=False):
        group = group.sort_values('ISO_TIME')
        lon = group['LON'].to_numpy(np.float64)
        lat = group['LAT'].to_numpy(np.float64)

        dt_h = group['ISO_TIME'].diff().dt.total_seconds().to_numpy() / 3600.0
        step_km = np.r_[0.0, haversine_distance(np.c_[lon[:-1], lat[:-1]], np.c_[lon[1:], lat[1:]])]
        gap = dt_h[1:] != step_hours
        jump = step_km[1:] > MAX_STEP_KM
        n_jump_cuts += int(np.sum(jump & ~gap))
        breaks = np.where(gap | jump)[0] + 1

        for seg_no, idx in enumerate(np.split(np.arange(len(group)), breaks)):
            if len(idx) < MIN_SEG_LEN:
                continue
            seg = group.iloc[idx]
            s_lon, s_lat = lon[idx], lat[idx]
            dir_rad = np.radians(seg['STORM_DIR'].to_numpy(np.float64))

            start_lon, start_lat = s_lon[0], s_lat[0]
            rel_lon = (s_lon - start_lon + 180.0) % 360.0 - 180.0
            rel_lat = s_lat - start_lat

            cols = [rel_lon, rel_lat, seg['STORM_SPEED'].to_numpy(np.float64),
                    np.sin(dir_rad), np.cos(dir_rad), s_lat]
            cyclones_data.append(np.column_stack(cols))
            initial_coords.append((start_lon, start_lat))

            name, year = str(seg['NAME'].iloc[0]).strip(), int(seg['YEAR'].iloc[0])
            meta.append({'id': storm_id, 'name': name, 'year': year, 'basin': str(seg['BASIN'].iloc[0]),
                         'nature': seg['NATURE'].astype(str).to_numpy(), 'seg': seg_no,
                         'label': f'{name}_{year}_ID{storm_id}_s{seg_no}'})
    print(f'Số chỗ cắt do nhảy > {MAX_STEP_KM:.0f} km: {n_jump_cuts}')
    return cyclones_data, initial_coords, meta

In [ ]:
all_cyc, all_init, meta = load_csv_data(CSV_PATH)
print(f'{len(all_cyc)} phân đoạn từ {len({m["id"] for m in meta})} cơn bão, '
      f'{sum(t.shape[0] for t in all_cyc)} điểm quỹ đạo')
pd.DataFrame(all_cyc[0], columns=['dlon', 'dlat', 'speed_kt', 'sin_dir', 'cos_dir', 'lat_abs']).head()

### 1.2 Chia tập train / val / test

Chia theo cơn bão (mọi phân đoạn của một cơn nằm cùng một tập), tỉ lệ 70/15/15 theo thứ tự ID như `split_by_storm` gốc.

In [ ]:
def split_by_storm(meta, f_train=0.7, f_val=0.85):
    ids = list(dict.fromkeys(m['id'] for m in meta))
    a, b = int(len(ids) * f_train), int(len(ids) * f_val)
    sets = [set(ids[:a]), set(ids[a:b]), set(ids[b:])]
    return [[i for i, m in enumerate(meta) if m['id'] in s] for s in sets]


split = split_by_storm(meta, 0.7, 0.85)
names = ['Huấn luyện', 'Kiểm định', 'Kiểm thử']

rows, id_sets = [], []
for name, idx in zip(names, split):
    ids = {meta[i]['id'] for i in idx}
    id_sets.append(ids)
    years = [meta[i]['year'] for i in idx]
    rows.append({'Tập': name, 'Số cơn bão': len(ids), 'Số phân đoạn': len(idx),
                 'Số điểm quỹ đạo': sum(all_cyc[i].shape[0] for i in idx),
                 **{f'Số cửa sổ +{6 * fs}h': sum(max(all_cyc[i].shape[0] - WINDOW_SIZE - fs, 0) for i in idx)
                    for fs in FORECAST_LIST},
                 'Năm': f'{min(years)}-{max(years)}'})
split_stats = pd.DataFrame(rows)
split_stats['Tỉ lệ cơn (%)'] = (100 * split_stats['Số cơn bão'] / split_stats['Số cơn bão'].sum()).round(2)
display(split_stats)

for a in range(3):
    for b in range(a + 1, 3):
        print(f'Trùng ID {names[a]} ∩ {names[b]}: {len(id_sets[a] & id_sets[b])} cơn')

## 2. Tạo cửa sổ đầu vào

Mỗi mẫu gồm:
- **X** (`WINDOW_SIZE` × số đặc trưng): mỗi hàng là một bước 6h. Cột 0, 1 của bảng được đổi thành độ dời km (Đông, Bắc) của bước đó; các cột khác lấy nguyên giá trị tại điểm đó.
- **Y** (1 × 2): độ dời km từ điểm neo *j* tới điểm *j + forecast_steps*.

Khác với bản gốc, `data_processing` nhận thêm `feature_cols` để chọn đặc trưng bất kỳ (ví dụ chỉ `[2, 3, 4]`), cần cho ablation.

In [ ]:
def deg_to_km(dlon, dlat, lat_ref):
    """Độ dời (độ) -> km (Đông, Bắc): dx = Δlon·cos(lat)·111, dy = Δlat·111."""
    coslat = np.cos(np.radians(lat_ref))
    return dlon * KM_PER_DEG * coslat, dlat * KM_PER_DEG


def km_to_deg(dx, dy, lat_ref):
    coslat = np.cos(np.radians(lat_ref))
    return dx / (KM_PER_DEG * coslat), dy / KM_PER_DEG


def anchor_indices(n_rows, window_size, forecast_steps=1):
    return np.arange(window_size, n_rows - forecast_steps)


def data_processing(table, window_size, forecast_steps=1, feature_cols=(0, 1, 2, 3, 4)):
    """Sinh (X, Y) theo cửa sổ trượt. Cột 0/1 trong feature_cols -> dx/dy (km); cột khác lấy nguyên."""
    table = np.asarray(table, dtype=np.float64)
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    nf = len(feature_cols)
    if len(js) == 0:
        return np.zeros((0, window_size, nf)), np.zeros((0, 1, 2))

    X = np.empty((len(js), window_size, nf))
    Y = np.empty((len(js), 1, 2))
    for n, j in enumerate(js):
        lat_ref = table[j, COL_LAT_ABS]
        steps = np.diff(table[j - window_size: j + 1, :2], axis=0)
        dx, dy = deg_to_km(steps[:, 0], steps[:, 1], lat_ref)
        for q, c in enumerate(feature_cols):
            X[n, :, q] = dx if c == 0 else dy if c == 1 else table[j - window_size + 1: j + 1, c]
        tgt = table[j + forecast_steps, :2] - table[j, :2]
        Y[n, 0] = deg_to_km(tgt[0], tgt[1], lat_ref)
    return X, Y


def persistence_km(table, window_size, forecast_steps=1):
    """Baseline persistence: lặp lại bước dời 6h gần nhất (km), tính thẳng từ bảng."""
    table = np.asarray(table, dtype=np.float64)
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    last = table[js, :2] - table[js - 1, :2]
    dx, dy = deg_to_km(last[:, 0], last[:, 1], table[js, COL_LAT_ABS])
    return forecast_steps * np.c_[dx, dy]

In [ ]:
X0, Y0 = data_processing(all_cyc[0], WINDOW_SIZE, 1)
print('X:', X0.shape, '| Y:', Y0.shape)
print('Cửa sổ đầu tiên (dx km, dy km, speed kt, sin, cos):\n', X0[0].round(2))
print('Nhãn +6h (dx, dy km):', Y0[0].round(2))

### 2.1 Speed / hướng tự tính từ tọa độ (Kinematics)

Dùng cho các cấu hình nhân quả trong ablation: giá trị tại điểm *k* chỉ dùng các điểm 0..*k*.
- `raw`: sai phân lùi.
- `kal`: bộ lọc Kalman nhân quả, trạng thái [x, y, vx, vy].
- `lag1`, `lag2`: lấy `STORM_SPEED`/`STORM_DIR` của IBTrACS nhưng lùi 1 hoặc 2 bước. Phần 7.1 cho thấy chỉ `lag2` mới hết rò rỉ.

In [ ]:
def step_km(lon, lat):
    """Độ dời km (Đông, Bắc) giữa các điểm liền kề, dùng vĩ độ trung điểm."""
    latm = np.radians((lat[1:] + lat[:-1]) / 2.0)
    dlon = (np.diff(lon) + 180.0) % 360.0 - 180.0
    return dlon * KM_PER_DEG * np.cos(latm), np.diff(lat) * KM_PER_DEG


def raw_velocity(lon, lat, dt=6.0):
    """Sai phân lùi (km/h). Hàng 0 chép từ hàng 1 (hàng 0 không bao giờ nằm trong cửa sổ)."""
    dx, dy = step_km(lon, lat)
    V = np.c_[dx, dy] / dt
    return np.vstack([V[:1], V])


def kalman_velocity(lon, lat, dt=6.0, sigma_pos=15.0, sigma_acc=0.5):
    """Kalman nhân quả; sigma_pos: sai số vị trí (km), sigma_acc: nhiễu gia tốc (km/h^2)."""
    dx, dy = step_km(lon, lat)
    pos = np.c_[np.r_[0.0, np.cumsum(dx)], np.r_[0.0, np.cumsum(dy)]]
    n = len(pos)
    A = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    Q1 = sigma_acc ** 2 * np.array([[dt ** 3 / 3, dt ** 2 / 2], [dt ** 2 / 2, dt]])
    Q = np.zeros((4, 4))
    Q[np.ix_([0, 2], [0, 2])] = Q1
    Q[np.ix_([1, 3], [1, 3])] = Q1
    H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)
    R = np.eye(2) * sigma_pos ** 2
    x = np.array([pos[0, 0], pos[0, 1], 0.0, 0.0])
    P = np.diag([sigma_pos ** 2, sigma_pos ** 2, 30.0 ** 2, 30.0 ** 2])
    V = np.zeros((n, 2))
    for k in range(1, n):
        x = A @ x
        P = A @ P @ A.T + Q
        K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
        x = x + K @ (pos[k] - H @ x)
        P = (np.eye(4) - K @ H) @ P
        V[k] = x[2:]
    V[0] = V[1]
    return V


def speed_sin_cos(V, eps=1e-6):
    """(tốc độ km/h, sin(hướng), cos(hướng)); hướng đo từ Bắc theo chiều kim đồng hồ."""
    sp = np.hypot(V[:, 0], V[:, 1])
    return sp, V[:, 0] / (sp + eps), V[:, 1] / (sp + eps)


def with_vel(t, mode, sigma_pos=15.0, sigma_acc=0.5):
    """Thay cột 2, 3, 4 bằng speed (knot), sin, cos tự tính từ tọa độ."""
    lon, lat = t[:, 0], t[:, COL_LAT_ABS]
    V = raw_velocity(lon, lat) if mode == 'raw' else kalman_velocity(lon, lat, sigma_pos=sigma_pos, sigma_acc=sigma_acc)
    sp, s, c = speed_sin_cos(V)
    t = t.copy()
    t[:, 2], t[:, 3], t[:, 4] = sp / 1.852, s, c
    return t


def lag_ib(t, k=2):
    """Lùi STORM_SPEED/DIR k bước. Phần 7.1: giá trị IBTrACS tại hàng r còn chứa thông tin tới vị trí r+2,
    nên phải lùi k = 2 mới hết rò rỉ (k = 1 vẫn rò rỉ)."""
    t = t.copy()
    t[k:, 2:5] = t[:-k, 2:5]
    return t

## 3. Mô hình MNN

Mỗi lớp là một lớp M-neuron song tuyến tính:

$$u_{b,j,k} = \sum_{p,q} X_{b,p,q}\, W^L_{j,k,p}\, W^R_{j,k,q} + \theta_{j,k}$$

Hai lớp ẩn có LayerNorm và hàm kích hoạt algebraic $x/\sqrt{1+x^2}$; lớp ra cho ma trận 1 × 2 (dx, dy đã chuẩn hoá).

In [ ]:
def algebraic_activation(x):
    return x / tf.sqrt(1.0 + tf.square(x))


def add_layer(inputs, in_shape, out_shape, activation_function=None, use_layernorm=False,
              layer_name='m_layer', drop_rate=None):
    n1, n2 = in_shape[1], in_shape[2]
    m1, m2 = out_shape[1], out_shape[2]
    init_std = (n1 * n2) ** -0.25          # giữ phương sai đầu ra của tích song tuyến tính ~ 1

    with tf.variable_scope(layer_name):
        W = tf.Variable(tf.random.normal([m1, m2, n1], stddev=init_std), name='W_L')
        V = tf.Variable(tf.random.normal([m1, m2, n2], stddev=init_std), name='W_R')
        biases = tf.Variable(tf.zeros([1, m1, m2]), name='theta')

        u = tf.einsum('bpq,jkp,jkq->bjk', inputs, W, V) + biases

        if use_layernorm:
            mean, variance = tf.nn.moments(u, axes=[1, 2], keepdims=True)
            gamma = tf.Variable(tf.ones([1, m1, m2]), name='gamma')
            beta = tf.Variable(tf.zeros([1, m1, m2]), name='beta')
            u = gamma * ((u - mean) / tf.sqrt(variance + 1e-6)) + beta

        outputs = u if activation_function is None else activation_function(u)
        if drop_rate is not None:
            outputs = tf.nn.dropout(outputs, rate=drop_rate)
    return W, V, outputs


def build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=None):
    _, _, l1 = add_layer(xs, [None, window_size, n_feat], first_hid,
                         activation_function=algebraic_activation, use_layernorm=True,
                         layer_name='layer1', drop_rate=drop_rate)
    _, _, l2 = add_layer(l1, first_hid, second_hid,
                         activation_function=algebraic_activation, use_layernorm=True,
                         layer_name='layer2', drop_rate=drop_rate)
    _, _, prediction = add_layer(l2, second_hid, [None, 1, 2], activation_function=None,
                                 use_layernorm=False, layer_name='layer3')
    return prediction

## 4. Huấn luyện

Giống `Training.py`: Adam, lr 1e-3, batch 256, clip gradient ±1, dừng sớm khi `stop_patience` epoch liền val không giảm quá 0,2%, lưu checkpoint tại val tốt nhất.

Hai thay đổi:
- Thư mục checkpoint mặc định nằm trong thư mục làm việc (notebook không có `__file__`).
- `lr_patience` trong bản gốc khai báo nhưng không dùng (lr luôn cố định). Ở đây để mặc định `None` (giữ nguyên hành vi cũ); đặt số nguyên để bật giảm lr ×0,5 khi val chững lại.

In [ ]:
def _stack(data, window_size, forecast_steps, feature_cols):
    Xs, Ys = [], []
    for table in data:
        X, Y = data_processing(table, window_size, forecast_steps, feature_cols)
        if len(X) > 0:
            Xs.append(X)
            Ys.append(Y)
    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)


def training(data_train, data_val, window_size, first_hid, second_hid, seed=None, forecast_steps=1,
             feature_cols=(0, 1, 2, 3, 4), save_dir=None, two_stage=False, batch_size=256,
             max_epochs=200, learning_rate=0.001, lr_patience=None, stop_patience=30,
             min_rel_improve=2e-3, min_lr=1e-5, verbose=True, drop_rate=0.0, l2_lambda=0.0):
    tf.reset_default_graph()
    seed = 0 if seed is None else seed
    tf.set_random_seed(seed)
    rng = np.random.RandomState(seed)

    X_tr, Y_tr = _stack(data_train, window_size, forecast_steps, feature_cols)
    X_va, Y_va = _stack(data_val, window_size, forecast_steps, feature_cols)
    n_feat = X_tr.shape[2]

    scaler_X, scaler_Y = StandardScaler(), StandardScaler()
    scaler_X.fit(X_tr.reshape(-1, n_feat))
    scaler_Y.fit(Y_tr.reshape(-1, 2))
    sx = lambda X: scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat).astype(np.float32)
    sy = lambda Y: scaler_Y.transform(Y.reshape(-1, 2)).reshape(-1, 1, 2).astype(np.float32)
    X_tr_s, Y_tr_s, X_va_s, Y_va_s = sx(X_tr), sy(Y_tr), sx(X_va), sy(Y_va)
    if verbose:
        print(f'   Mẫu train: {len(X_tr_s)} | mẫu val: {len(X_va_s)} | số đặc trưng: {n_feat}')

    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    ys = tf.placeholder(tf.float32, [None, 1, 2])
    lr_ph = tf.placeholder(tf.float32, [])
    drop_ph = tf.placeholder_with_default(0.0, [], name='drop_rate') if drop_rate > 0 else None
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=drop_ph)
    loss = tf.reduce_mean(tf.square(ys - prediction))
    if l2_lambda > 0:
        l2 = tf.add_n([tf.nn.l2_loss(v) for v in tf.trainable_variables() if 'W_L' in v.name or 'W_R' in v.name])
        train_loss = loss + l2_lambda * l2
    else:
        train_loss = loss

    def make_train_op(var_list):
        opt = tf.train.AdamOptimizer(learning_rate=lr_ph)
        gv = opt.compute_gradients(train_loss, var_list)
        gv = [(tf.clip_by_value(g, -1.0, 1.0), v) for g, v in gv if g is not None]
        return opt.apply_gradients(gv)

    all_vars = tf.trainable_variables()
    if two_stage:
        train_ops = [make_train_op([v for v in all_vars if 'W_R' not in v.name]),
                     make_train_op([v for v in all_vars if 'W_R' in v.name])]
    else:
        train_ops = [make_train_op(all_vars)]

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, 'model')

    sess = tf.Session()
    sess.run(tf.global_variables_initializer())
    saver = tf.train.Saver(tf.trainable_variables())

    def full_loss(X, Y, chunk=8192):
        tot = 0.0
        for i in range(0, len(X), chunk):
            tot += sess.run(loss, {xs: X[i:i + chunk], ys: Y[i:i + chunk]}) * len(X[i:i + chunk])
        return tot / len(X)

    train_losses, val_losses = [], []
    best_val, best_epoch, lr = np.inf, -1, learning_rate
    ref_val, since_best, since_lr = np.inf, 0, 0
    n = len(X_tr_s)

    for epoch in range(max_epochs):
        perm = rng.permutation(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            fd = {xs: X_tr_s[b], ys: Y_tr_s[b], lr_ph: lr}
            if drop_ph is not None:
                fd[drop_ph] = drop_rate
            for op in train_ops:
                sess.run(op, fd)

        tr_l, va_l = full_loss(X_tr_s, Y_tr_s), full_loss(X_va_s, Y_va_s)
        train_losses.append(tr_l)
        val_losses.append(va_l)

        if va_l < best_val:
            best_val, best_epoch = va_l, epoch
            saver.save(sess, save_path)

        if va_l < ref_val * (1.0 - min_rel_improve):
            ref_val, since_best, since_lr = va_l, 0, 0
        else:
            since_best += 1
            since_lr += 1
            if lr_patience is not None and since_lr >= lr_patience and lr > min_lr:
                lr, since_lr = max(lr * 0.5, min_lr), 0
                if verbose:
                    print(f'   epoch {epoch}: giảm lr xuống {lr:.1e}')
            if since_best >= stop_patience:
                if verbose:
                    print(f'   Dừng sớm ở epoch {epoch}: {stop_patience} epoch liền val không giảm quá '
                          f'{100 * min_rel_improve:.1f}%')
                break

        if verbose and epoch % 10 == 0:
            print(f'   epoch {epoch:3d}  train {tr_l:.5f}  val {va_l:.5f}  lr {lr:.1e}')

    sess.close()
    if verbose:
        print(f'   Best val loss {best_val:.5f} tại epoch {best_epoch}')
    return scaler_X, scaler_Y, train_losses, val_losses, best_epoch

## 5. Kiểm thử

Khôi phục checkpoint tốt nhất, dự báo từng phân đoạn và đổi ngược về km.

In [ ]:
def testing(data_test, window_size, first_hid, second_hid, scaler_X, scaler_Y, forecast_steps=1,
            feature_cols=(0, 1, 2, 3, 4), save_dir=None):
    """Trả về (Y_true_km, Y_pred_km, valid_indices)."""
    tf.reset_default_graph()
    n_feat = scaler_X.n_features_in_
    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid)

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    sess = tf.Session()
    tf.train.Saver(tf.trainable_variables()).restore(sess, os.path.join(save_dir, 'model'))

    Y_true, Y_pred, valid = [], [], []
    for idx, storm in enumerate(data_test):
        X, Y = data_processing(storm, window_size, forecast_steps, feature_cols)
        if len(X) == 0:
            continue
        valid.append(idx)
        X_s = scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat)
        p = sess.run(prediction, feed_dict={xs: X_s})
        Y_true.append(Y.reshape(-1, 2))
        Y_pred.append(scaler_Y.inverse_transform(p.reshape(-1, 2)))
    sess.close()
    return Y_true, Y_pred, valid


def to_abs(table, initial, disp_km, forecast_steps):
    """Độ dời km từ điểm neo -> toạ độ tuyệt đối (lon, lat)."""
    js = anchor_indices(table.shape[0], WINDOW_SIZE, forecast_steps)
    anchor = table[js, :2] + np.array(initial)
    dlon, dlat = km_to_deg(disp_km[:, 0], disp_km[:, 1], table[js, COL_LAT_ABS])
    return anchor + np.c_[dlon, dlat]


def hav_errors(tables, initials, valid, true_km, pred_km, fs):
    """Sai số Haversine (km) của mọi điểm dự báo, nối theo thứ tự phân đoạn."""
    return np.concatenate([haversine_distance(to_abs(tables[i], initials[i], t, fs),
                                              to_abs(tables[i], initials[i], p, fs))
                           for i, t, p in zip(valid, true_km, pred_km)])

## 6. Quét hệ số L2

Chỉnh danh sách λ, seed và cấu hình đầu vào ở ô dưới.

| Cấu hình | Đầu vào | Nhân quả |
|---|---|---|
| A | dx, dy | có |
| C | dx, dy + STORM_SPEED/DIR gốc của IBTrACS (như `MNN_Main`) | **không** (rò rỉ vị trí bước kế tiếp) |
| D | dx, dy + STORM_SPEED/DIR trễ 1 bước | **không** (vẫn rò rỉ, xem MNN_TongHop phần 7.1) |
| D2 | dx, dy + STORM_SPEED/DIR trễ 2 bước | có |

Mặc định chạy D2. Thời gian ước tính: số λ × số seed × số cấu hình × 3–5 phút (khi `QUICK = False`).

In [ ]:
L2_LIST = [0.0, 5e-4, 1e-3, 2e-3, 5e-3]   # λ = 0 là mốc so sánh, nên luôn giữ
L2_SEEDS = [42, 43, 44]
L2_CFGS = ['D2']                           # thêm 'A', 'C', 'D' nếu muốn, ví dụ ['A', 'D2']
L2_FS = 1                                  # 1 -> +6h, 2 -> +12h
L2_DROP = 0.0                              # dropout giữ cố định cho mọi λ
L2_DIR = 'KetQua_L2'
N_BOOT = 2000

CFGS = {
    'A': ([0, 1],          'ib',  True,  'A: Δlon, Δlat'),
    'C': ([0, 1, 2, 3, 4], 'ib',  False, 'C: 5 đặc trưng, speed/dir IBTrACS gốc'),
    'D':  ([0, 1, 2, 3, 4], 'lag1', False, 'D: 5 đặc trưng, speed/dir IBTrACS trễ 1 bước (vẫn rò rỉ)'),
    'D2': ([0, 1, 2, 3, 4], 'lag2', True,  'D2: 5 đặc trưng, speed/dir IBTrACS trễ 2 bước'),
}

if QUICK:
    L2_SEEDS = [42]

os.makedirs(L2_DIR, exist_ok=True)


def make_source(mode):
    if mode == 'ib':
        return all_cyc
    if mode.startswith('lag'):
        return [lag_ib(t, int(mode[3:] or 1)) for t in all_cyc]
    return [with_vel(t, mode) for t in all_cyc]


SRC = {mode: make_source(mode) for mode in {CFGS[c][1] for c in L2_CFGS}}
tr_idx, va_idx, te_idx = split
print(f'{len(L2_LIST)} giá trị λ × {len(L2_SEEDS)} seed × {len(L2_CFGS)} cấu hình = '
      f'{len(L2_LIST) * len(L2_SEEDS) * len(L2_CFGS)} lần chạy, tối đa {MAX_EPOCHS} epoch mỗi lần')

### 6.1 Huấn luyện

In [ ]:
def l2_file(cfg, lam, seed):
    return os.path.join(L2_DIR, f'{cfg}_l2_{lam:g}_fs{L2_FS}_ep{MAX_EPOCHS}_s{seed}.npz')


def run_one(cfg, lam, seed):
    path = l2_file(cfg, lam, seed)
    if os.path.exists(path):
        return False
    cols, src, _, _ = CFGS[cfg]
    T = SRC[src]
    tr, va, te = ([T[i] for i in idx] for idx in split)
    va_init, te_init = [all_init[i] for i in va_idx], [all_init[i] for i in te_idx]
    sd = os.path.join(L2_DIR, 'ckpt', f'{cfg}_{lam:g}_{seed}')
    t0 = time.time()
    sX, sY, trl, vll, best = training(tr, va, WINDOW_SIZE, FIRST_HID, SECOND_HID, seed=seed, forecast_steps=L2_FS,
                                      feature_cols=cols, save_dir=sd, two_stage=TWO_STAGE, max_epochs=MAX_EPOCHS,
                                      stop_patience=STOP_PATIENCE, drop_rate=L2_DROP, l2_lambda=lam, verbose=False)
    out = {'best_epoch': best, 'train_losses': np.array(trl), 'val_losses': np.array(vll)}
    for name, tabs, inits in (('val', va, va_init), ('test', te, te_init)):
        yt, yp, valid = testing(tabs, WINDOW_SIZE, FIRST_HID, SECOND_HID, sX, sY, L2_FS, cols, save_dir=sd)
        out[f'err_{name}'] = hav_errors(tabs, inits, valid, yt, yp, L2_FS)
    np.savez(path, **out)
    shutil.rmtree(sd, ignore_errors=True)
    ev, et = out['err_val'], out['err_test']
    print(f'val Mean {ev.mean():6.2f} RMSE {np.sqrt(np.mean(ev ** 2)):6.2f} | '
          f'test Mean {et.mean():6.2f} RMSE {np.sqrt(np.mean(et ** 2)):6.2f} km | '
          f'best_epoch {best} | {time.time() - t0:.0f}s')
    return True


jobs = [(c, lam, s) for c in L2_CFGS for s in L2_SEEDS for lam in L2_LIST]
for n, (c, lam, s) in enumerate(jobs, 1):
    print(f'[{n}/{len(jobs)}] {c} λ={lam:g} seed={s}: ', end='')
    if not run_one(c, lam, s):
        print('đã có, bỏ qua')

### 6.2 Bảng kết quả theo λ

- **Val Mean / Val RMSE**: dùng để chọn λ (đánh dấu ★ ở cột "Chọn theo val").
- **Test**: Mean, RMSE (kèm độ lệch chuẩn theo seed), Median, P95, skill so với persistence.
- **gap** = val loss − train loss tại best epoch (MSE chuẩn hoá). Gap lớn và dương là dấu hiệu quá khớp; L2 chỉ có ích khi có quá khớp.
- **Bảng thứ hai**: Δ = (λ đang xét) − (λ = 0) trên test, CI95 bootstrap theo cơn bão. Δ < 0 nghĩa là L2 giúp giảm sai số; "CI loại trừ 0" nghĩa là khác biệt có ý nghĩa thống kê.

In [ ]:
def rmse(e):
    return float(np.sqrt(np.mean(e ** 2)))


def test_groups(fs):
    """Mã cơn bão của từng điểm test (đúng thứ tự hav_errors) và sai số persistence."""
    te = [all_cyc[i] for i in te_idx]
    ids = np.concatenate([[meta[i]['id']] * len(anchor_indices(all_cyc[i].shape[0], WINDOW_SIZE, fs))
                          for i in te_idx])
    true_km = [data_processing(t, WINDOW_SIZE, fs, (0, 1))[1].reshape(-1, 2) for t in te]
    per_km = [persistence_km(t, WINDOW_SIZE, fs) for t in te]
    e_per = hav_errors(te, [all_init[i] for i in te_idx], range(len(te)), true_km, per_km, fs)
    return pd.factorize(ids)[0], e_per


def boot_delta(Ex, Ey, groups, n_boot=N_BOOT, seed=SEED):
    """Δ Mean, Δ RMSE (X − Y, trung bình qua seed) và CI95 bootstrap theo cơn bão. Ex, Ey: (seed, điểm)."""
    G = groups.max() + 1
    cnt = np.bincount(groups, minlength=G).astype(float)
    S1 = lambda E: np.stack([np.bincount(groups, e, G) for e in E])
    S2 = lambda E: np.stack([np.bincount(groups, e ** 2, G) for e in E])
    s1x, s2x, s1y, s2y = S1(Ex), S2(Ex), S1(Ey), S2(Ey)
    w = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=n_boot).astype(float)
    nb = w @ cnt
    d_mean = ((w @ s1x.T) - (w @ s1y.T)).mean(axis=1) / nb
    d_rmse = (np.sqrt((w @ s2x.T) / nb[:, None]) - np.sqrt((w @ s2y.T) / nb[:, None])).mean(axis=1)
    pm = np.mean(Ex.mean(axis=1) - Ey.mean(axis=1))
    pr = np.mean(np.sqrt((Ex ** 2).mean(axis=1)) - np.sqrt((Ey ** 2).mean(axis=1)))
    return pm, np.percentile(d_mean, [2.5, 97.5]), pr, np.percentile(d_rmse, [2.5, 97.5])


groups, e_per = test_groups(L2_FS)
l2_results = {}
for cfg in L2_CFGS:
    rows, E = [], {}
    for lam in L2_LIST:
        files = [l2_file(cfg, lam, s) for s in L2_SEEDS if os.path.exists(l2_file(cfg, lam, s))]
        if not files:
            continue
        runs = [np.load(f) for f in files]
        et = np.stack([r['err_test'] for r in runs])
        ev = np.stack([r['err_val'] for r in runs])
        E[lam] = et
        sd = (lambda v: v.std(ddof=1)) if len(runs) > 1 else (lambda v: np.nan)
        t_mean, t_rmse = et.mean(axis=1), np.sqrt((et ** 2).mean(axis=1))
        gaps = [r['val_losses'][int(r['best_epoch'])] - r['train_losses'][int(r['best_epoch'])] for r in runs]
        rows.append({'L2 λ': lam, 'n_seed': len(runs),
                     'Val Mean (km)': ev.mean(axis=1).mean(), 'Val RMSE (km)': np.sqrt((ev ** 2).mean(axis=1)).mean(),
                     'Test Mean (km)': t_mean.mean(), 'Std Test Mean': sd(t_mean),
                     'Test RMSE (km)': t_rmse.mean(), 'Std Test RMSE': sd(t_rmse),
                     'Test Median (km)': np.median(et, axis=1).mean(),
                     'Test P95 (km)': np.percentile(et, 95, axis=1).mean(),
                     'best_epoch': np.mean([int(r['best_epoch']) for r in runs]), 'gap': np.mean(gaps)})
    if not rows:
        continue
    tab = pd.DataFrame(rows)
    tab['Skill Mean vs persistence (%)'] = 100 * (1 - tab['Test Mean (km)'] / e_per.mean())
    tab['Skill RMSE vs persistence (%)'] = 100 * (1 - tab['Test RMSE (km)'] / rmse(e_per))
    tab['Chọn theo val'] = ''
    tab.loc[tab['Val Mean (km)'].idxmin(), 'Chọn theo val'] += '★ Mean '
    tab.loc[tab['Val RMSE (km)'].idxmin(), 'Chọn theo val'] += '★ RMSE'
    tab['L2 λ'] = tab['L2 λ'].map(lambda v: f'{v:g}')

    comp = []
    if 0.0 in E:
        for lam in L2_LIST:
            if lam == 0.0 or lam not in E:
                continue
            dm, ci_m, dr, ci_r = boot_delta(E[lam], E[0.0], groups)
            comp.append({'So sánh': f'λ={lam:g} so với λ=0', 'Δ Mean (km)': dm,
                         'CI95 Mean': f'[{ci_m[0]:.2f}, {ci_m[1]:.2f}]',
                         'Mean: CI loại trừ 0': bool(ci_m[0] > 0 or ci_m[1] < 0),
                         'Δ RMSE (km)': dr, 'CI95 RMSE': f'[{ci_r[0]:.2f}, {ci_r[1]:.2f}]',
                         'RMSE: CI loại trừ 0': bool(ci_r[0] > 0 or ci_r[1] < 0)})
    comp = pd.DataFrame(comp)
    tab.round(4).to_csv(os.path.join(L2_DIR, f'Bang_L2_{cfg}_{6 * L2_FS}h.csv'), index=False, encoding='utf-8-sig')
    comp.round(4).to_csv(os.path.join(L2_DIR, f'So_sanh_L2_{cfg}_{6 * L2_FS}h.csv'), index=False, encoding='utf-8-sig')
    l2_results[cfg] = (tab, comp)

    print(f'\n========== {CFGS[cfg][3]} | +{6 * L2_FS}h | {MAX_EPOCHS} epoch tối đa ==========')
    print(f'Persistence (test): Mean {e_per.mean():.2f} | RMSE {rmse(e_per):.2f} | '
          f'Median {np.median(e_per):.2f} | P95 {np.percentile(e_per, 95):.2f} km')
    display(tab.set_index('L2 λ').round(3))
    if len(comp):
        display(comp.set_index('So sánh').round(3))
print(f'Đã lưu CSV vào {L2_DIR}/')

### 6.3 Biểu đồ sai số theo λ và đường cong loss

Hàng trên: Mean và RMSE trên val (dùng để chọn λ) và test, thanh lỗi là độ lệch chuẩn theo seed, vạch đứt là persistence (test).
Hàng dưới: val loss theo epoch của seed đầu tiên, màu càng đậm thì λ càng lớn.

In [ ]:
for cfg, (tab, _) in l2_results.items():
    lams = list(tab['L2 λ'])
    x = np.arange(len(lams))
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, metric in zip(axes[0], ['Mean', 'RMSE']):
        ax.plot(x, tab[f'Val {metric} (km)'], marker='o', ms=8, lw=2, color='#9DB9D9', label='Val')
        ax.errorbar(x, tab[f'Test {metric} (km)'], yerr=tab[f'Std Test {metric}'].fillna(0), marker='o', ms=8,
                    lw=2, capsize=4, color='#2F5E8E', label='Test')
        ax.axhline(e_per.mean() if metric == 'Mean' else rmse(e_per), color='#555555', ls='--', lw=1.5,
                   label='Persistence (test)')
        ax.set_xticks(x, lams)
        ax.set_xlabel('L2 λ')
        ax.set_ylabel('km')
        ax.set_title(f'{metric} sai số Haversine +{6 * L2_FS}h')
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(frameon=False)

    cmap = plt.get_cmap('Blues')
    for ax, key, title in zip(axes[1], ['train_losses', 'val_losses'], ['Train loss', 'Val loss']):
        for k, lam in enumerate(L2_LIST):
            f = l2_file(cfg, lam, L2_SEEDS[0])
            if os.path.exists(f):
                ax.plot(np.load(f)[key], lw=2, color=cmap(0.35 + 0.65 * k / max(len(L2_LIST) - 1, 1)),
                        label=f'λ={lam:g}')
        ax.set_title(f'{title} (MSE chuẩn hoá), seed {L2_SEEDS[0]}')
        ax.set_xlabel('epoch')
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(frameon=False, fontsize=10)
    fig.suptitle(CFGS[cfg][3])
    fig.tight_layout()
    plt.show()

### 6.4 Cách đọc

- Nếu **gap ≈ 0 hoặc âm** ở λ = 0 thì mô hình không quá khớp, L2 khó có ích; λ lớn sẽ kéo cả train lẫn val loss lên (thiếu khớp).
- Chọn λ có **Val Mean/RMSE nhỏ nhất**; chỉ coi là cải thiện thật khi Δ trên test âm **và** CI95 loại trừ 0.
- Kết quả với `QUICK = True` (4 epoch) chỉ cho thấy xu hướng đầu quá trình huấn luyện, chưa phản ánh tác dụng chống quá khớp của L2.